# Multiple Linear Regression


## import the libraries


In [55]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Import the dataset

In [56]:
from re import X
df = pd.read_csv('50_Startups.csv')
# X is the matrix of the feature
X = df.iloc[:, :-1].values
# this is just the vector. Not the matrix. this is dependent variable vaector
# why not range creation..?
y = df.iloc[:, -1].values

## Importing the catagorical independent variable

In [57]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers = [('encoder', OneHotEncoder(), [3])], remainder = 'passthrough')
X = np.array(ct.fit_transform(X))

## Spliting the dataset into test set and training set

In [58]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

## Training the multiple linear regression model on the training set

In [59]:
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(X_train, y_train)

LinearRegression()

## Predicting the test set result

In [60]:
y_pred = regressor.predict(X_test)

In [61]:
np.set_printoptions(precision=2)
np.concatenate((y_test.reshape(len(y_test),1), y_pred.reshape(len(y_pred),1)), axis = 1)

array([[134307.35, 126362.88],
       [ 81005.76,  84608.45],
       [ 99937.59,  99677.49],
       [ 64926.08,  46357.46],
       [125370.37, 128750.48],
       [ 35673.41,  50912.42],
       [105733.54, 109741.35],
       [107404.34, 100643.24],
       [ 97427.84,  97599.28],
       [122776.86, 113097.43]])

## Building the multiple linear regrassion machine learning model on the full dataset using the sklearn library


In [62]:
regressor_full = LinearRegression()
regressor_full.fit(X, y)

LinearRegression()

In [63]:
print(regressor_full.coef_)

[-5.23e+01  1.46e+02 -9.42e+01  8.06e-01 -2.70e-02  2.70e-02]


In [64]:
print(regressor_full.intercept_)

50177.64442290799


$$\textrm{Profit} =  50177.64 - 52.3 \times \textrm{Dummy State 1} + 146.49 \times \textrm{Dummy State 2} - 94.19 \times \textrm{Dummy State 3} + 0.806 \times \textrm{R&D Spend} - 0.027 \times \textrm{Administration} + 0.0269 \times \textrm{Marketing Spend}$$

## Making a single prediction (for example the profit of a startup with R&D Spend = 160000, Administration Spend = 130000, Marketing Spend = 300000 and State = 'California')


In [65]:
print(regressor_full.predict([[1, 0, 0, 160000, 130000, 300000]]))

[183672.44]


In [66]:
Result = 50177.64 - 52.3*1 + 146.49*0 - 94.19*0 + 0.806*160000 - 0.027 * 130000 + 0.0269 * 300000
print(Result)

183645.34000000003


## Building the multiple linear regression machine learning model using the statesmodels.api


### Avoiding Dummay variable trap

* The dummy variable are highly corelated. when one column is predicting the other column. That's why suggestted to reduce the dummy variable. so the trap is to remain all the dummy variable.
If there is 3 or more dummy varibale are present then we have to exclue one dummy varibale.

* Sklearn did this autometically.

- But we didn't get the full statistical analysis of the model and data. that's why we are now going to build the model with stats.api


In [67]:
X_next = X[:, 1:]

### Dealing with the constant of the model

y = b0x0 -> need to define the x0

In [68]:
ones = np.ones((50,1)).astype(int)
X_next = np.append(ones, X_next, axis = 1)

### Building the model


In [69]:
import statsmodels.api as sm

Needs to convert all the feature into the float. then the statsapi can run.

In [70]:
X_opt = X_next.astype(np.float64)

In [71]:
regressor_OLS = sm.OLS(endog = y, exog = X_opt).fit()
regressor_OLS.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.951
Model:                            OLS   Adj. R-squared:                  0.945
Method:                 Least Squares   F-statistic:                     169.9
Date:                Wed, 11 Feb 2026   Prob (F-statistic):           1.34e-27
Time:                        17:31:00   Log-Likelihood:                -525.38
No. Observations:                  50   AIC:                             1063.
Df Residuals:                      44   BIC:                             1074.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       5.013e+04   6884.820      7.281      0.000    3.62e+04     6.4e+04
x1           198.7888   3371.007      0.059      0.953   -6595.030    6992.607
x2           -41.8870   3256.039     -0.013      0.990   -6604.003    6520.229
x3             0.8060      0.046     17.369      0.000       0.712       0.900
x4            -0.0270      0.052     -0.517      0.608      -0.132       0.078
x5             0.0270      0.017      1.574      0.123      -0.008       0.062
==============================================================================
Omnibus:                       14.782   Durbin-Watson:                   1.283
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               21.266
Skew:                          -0.948   Prob(JB):                     2.41e-05
Kurtosis:                       5.572   Cond. No.                     1.45e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.45e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [72]:
Result2 = 50130 + 198.7888*0 - 41.8870*0 + 0.8060*160000 - 0.0270 * 130000 + 0.0270 * 300000
print(Result2)

183680.0


## Building the optimal model using the statsmodel.api

Backward elimination technique

In [73]:
X_opt = X_next[ : ,[0, 3, 4, 5] ].astype(np.float64)
regressor_OLS = sm.OLS(endog = y, exog = X_opt).fit()
regressor_OLS.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.951
Model:                            OLS   Adj. R-squared:                  0.948
Method:                 Least Squares   F-statistic:                     296.0
Date:                Wed, 11 Feb 2026   Prob (F-statistic):           4.53e-30
Time:                        17:31:00   Log-Likelihood:                -525.39
No. Observations:                  50   AIC:                             1059.
Df Residuals:                      46   BIC:                             1066.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       5.012e+04   6572.353      7.626      0.000    3.69e+04    6.34e+04
x1             0.8057      0.045     17.846      0.000       0.715       0.897
x2            -0.0268      0.051     -0.526      0.602      -0.130       0.076
x3             0.0272      0.016      1.655      0.105      -0.006       0.060
==============================================================================
Omnibus:                       14.838   Durbin-Watson:                   1.282
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               21.442
Skew:                          -0.949   Prob(JB):                     2.21e-05
Kurtosis:                       5.586   Cond. No.                     1.40e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.4e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [74]:
X_opt = X_next[ : ,[0, 3, 5] ].astype(np.float64)
regressor_OLS = sm.OLS(endog = y, exog = X_opt).fit()
regressor_OLS.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.950
Model:                            OLS   Adj. R-squared:                  0.948
Method:                 Least Squares   F-statistic:                     450.8
Date:                Wed, 11 Feb 2026   Prob (F-statistic):           2.16e-31
Time:                        17:31:00   Log-Likelihood:                -525.54
No. Observations:                  50   AIC:                             1057.
Df Residuals:                      47   BIC:                             1063.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4.698e+04   2689.933     17.464      0.000    4.16e+04    5.24e+04
x1             0.7966      0.041     19.266      0.000       0.713       0.880
x2             0.0299      0.016      1.927      0.060      -0.001       0.061
==============================================================================
Omnibus:                       14.677   Durbin-Watson:                   1.257
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               21.161
Skew:                          -0.939   Prob(JB):                     2.54e-05
Kurtosis:                       5.575   Cond. No.                     5.32e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 5.32e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""